##MODELO

In [0]:
from pyspark.sql.functions import col
from pyspark.sql.functions import when
from pyspark.sql.functions import col, isnull, isnan, sum as spark_sum
from pyspark.sql.functions import to_date
from pyspark.sql.functions import coalesce
from pyspark.sql import functions as F
from pyspark.sql.functions import count, avg
from pyspark.sql import SparkSession
from pyspark.sql.window import Window
from pyspark.sql.functions import col, row_number
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from pyspark.sql.functions import col, mean as _mean, stddev as _stddev
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from functools import reduce

In [0]:
# Preparar para o modelo
columns_to_model = [    
    'gender_encoded',
    'target_encoded',
    'age_category',
    'credit_card_limit_category',
    'total_transactions_category',
    'days_since_registration_category',
    'engagement_mobile_channel_category',
    'engagement_web_channel_category',
    'engagement_social_channel_category',
    'avg_min_value_category',
    'avg_offer_duration_category',
    'avg_discount_value_category'
]

df_model = df_individuo[columns_to_model]


# 1. Lista de variáveis categóricas
categorical_columns = [
       'gender_encoded',
       'age_category',
       'credit_card_limit_category',
       'total_transactions_category',
       'days_since_registration_category',
       'engagement_mobile_channel_category',
       'engagement_web_channel_category',
       'engagement_social_channel_category',
       'avg_min_value_category',
       'avg_offer_duration_category',
       'avg_discount_value_category'
   ]

X = df_model.select(feature_cols)
y = df_model.select('target_encoded')

# 2. Converter para Pandas
X_pandas = X.toPandas()  # Converte features para Pandas
y_pandas = y.toPandas()['target_encoded']  # Ajustar target (coluna)

# Checar e tratar valores ausentes (NaN)
X_pandas = X_pandas.dropna()  # Remove valores ausentes
y_pandas = y_pandas[X_pandas.index]  # Alinha índices de y com X

# Realizar o One-Hot Encoding (Criação de Dummies)
X_pandas = pd.get_dummies(X_pandas, columns=categorical_columns, drop_first=True)

# 3. Divisão entre Treinamento e Teste
X_train, X_test, y_train, y_test = train_test_split(
    X_pandas,
    y_pandas,
    test_size=0.3,
    random_state=42,
    stratify=y_pandas  # Preservar distribuição das classes
)

# 4. Configurar e ajustar o modelo multinomial
lr = LogisticRegression(multi_class='multinomial', solver='lbfgs', max_iter=500)
lr.fit(X_train, y_train)  # Ajustar o modelo nos dados de treino

# 5. Fazer as previsões
y_pred = lr.predict(X_test)

# 6. Avaliar o desempenho do modelo
# Acurácia
acc = accuracy_score(y_test, y_pred)
# F1-Score ponderado
f1 = f1_score(y_test, y_pred, average='weighted')
# Matriz de Confusão
cm = confusion_matrix(y_test, y_pred)

# 7. Imprimir os resultados
print(f'Acurácia: {acc:.4f}')
print(f'F1-Score: {f1:.4f}')
print('Matriz de confusão:')
print(cm)

In [0]:
# 1. Adicionar a constante (intercepto) aos dados
X_pandas = sm.add_constant(X_pandas)

# 2. Ajustar o modelo multinomial usando MNLogit
model = sm.MNLogit(y_pandas, X_pandas)
result = model.fit(method='newton')  # Método para maximizar a verossimilhança

# 3. Exibir o resumo do modelo
print(result.summary())

# 4. Obter os coeficientes (odds ratio se necessário)
print("\nCoeficientes:")
print(result.params)  # Coeficientes do modelo

# 5. Intervalos de confiança (95% por padrão)
print("\nIntervalos de Confiança dos Coeficientes:")
print(result.conf_int())

# 6. p-Valores
print("\nP-Valores dos Coeficientes:")
print(result.pvalues)


In [0]:
# 4. Obter os coeficientes (odds ratio se necessário)
print("\nCoeficientes (log-odds):")
print(result.params)  # Coeficientes do modelo

# Coeficientes exponenciados (odds ratios)
print("\nOdds Ratios (exp(coeficientes)):")
print(np.exp(result.params))

# 5. Intervalos de confiança (95% por padrão)
conf = result.conf_int()
print("\nIntervalos de Confiança dos Coeficientes (log-odds):")
print(conf)

# Intervalos de confiança exponenciados (odds ratios)
print("\nIntervalos de Confiança dos Odds Ratios:")
print(np.exp(conf))

# 6. p-Valores
print("\nP-Valores dos Coeficientes:")
print(result.pvalues)

In [0]:
# Verificar se há colunas constantes (sem variação)
print(X_pandas.var())

In [0]:
# Prever as probabilidades para o conjunto de teste
y_probabilities = lr.predict_proba(X_test)

# Transformar em um DataFrame para facilitar a visualização
probabilities_df = pd.DataFrame(
    y_probabilities,
    columns=[f"Probabilidade_target_{i}" for i in range(y_probabilities.shape[1])],
    index=X_test.index  # Manter o mesmo índice dos dados de teste
)

# Exibir as probabilidades
probabilities_df.display()

In [0]:
corr_matrix = X_pandas.corr()
display(corr_matrix)